# Regime-RL Trading – Exploration Notebook

This notebook walks through:
1. Generating (or loading) synthetic OHLCV data
2. Computing technical features with `FeatureEngineer`
3. Detecting market regimes with `FeatureRegimeDetector`
4. Visualising regimes on a price chart
5. Running a single episode in `TradingEnv`
6. Plotting a quick equity curve

In [ ]:
import sys, os
# Make sure the project root is on the path
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Synthetic OHLCV data

In [ ]:
def make_synthetic_ohlcv(n=500, seed=0):
    rng = np.random.default_rng(seed)
    # Mix of trending and mean-reverting periods
    changes = np.concatenate([
        rng.normal(0.002, 0.008, 125),   # bull
        rng.normal(-0.001, 0.015, 125),  # volatile bear
        rng.normal(0.0, 0.004, 125),     # sideways
        rng.normal(0.003, 0.007, 125),   # bull
    ])
    close  = 100.0 * np.exp(np.cumsum(changes))
    high   = close * (1 + rng.uniform(0, 0.015, n))
    low    = close * (1 - rng.uniform(0, 0.015, n))
    open_  = low + rng.uniform(0, 1, n) * (high - low)
    volume = rng.integers(1_000_000, 5_000_000, n).astype(float)
    return pd.DataFrame({'Open': open_, 'High': high, 'Low': low,
                         'Close': close, 'Volume': volume})

data = make_synthetic_ohlcv()
print(f'Shape: {data.shape}')
data.head()

## 2. Feature Engineering

In [ ]:
from src.environment.features import FeatureEngineer, FEATURE_NAMES

eng = FeatureEngineer()
features = eng.compute(data)
print('Features shape:', features.shape)
print('Feature names:', FEATURE_NAMES)
features.describe()

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 14))
for ax, col in zip(axes.ravel(), FEATURE_NAMES):
    ax.plot(features[col], linewidth=0.8)
    ax.set_title(col)
    ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
plt.suptitle('Technical Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Regime Detection

In [ ]:
from src.regime_detection.feature_detector import FeatureRegimeDetector
from src.regime_detection.base import MarketRegime

detector = FeatureRegimeDetector()
regimes = detector.predict(data)

from collections import Counter
print('Regime distribution:', {r.name: v for r, v in Counter(regimes).items()})

In [ ]:
COLORS = {
    MarketRegime.BULL:     '#2ca02c',
    MarketRegime.BEAR:     '#d62728',
    MarketRegime.SIDEWAYS: '#ff7f0e',
    MarketRegime.VOLATILE: '#9467bd',
}

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(data['Close'].values, color='steelblue', linewidth=1)
for i, regime in enumerate(regimes):
    ax.axvspan(i, i + 1, alpha=0.2, color=COLORS[regime], linewidth=0)
patches = [mpatches.Patch(color=c, alpha=0.5, label=r.name) for r, c in COLORS.items()]
ax.legend(handles=patches, loc='upper left')
ax.set_title('Price with Regime Overlay')
ax.set_xlabel('Day')
ax.set_ylabel('Price ($)')
plt.tight_layout()
plt.show()

## 4. Run a Random-Agent Episode in TradingEnv

In [ ]:
from src.environment.trading_env import TradingEnv

env = TradingEnv(data, lookback_window=20)
obs, _ = env.reset(seed=0)
print('Observation shape:', obs.shape)
print('Action space      :', env.action_space)

In [ ]:
portfolio_values = [env.initial_cash]
episode_regimes  = []

done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    portfolio_values.append(info['portfolio_value'])
    episode_regimes.append(info['regime'])
    done = terminated or truncated

print(f'Episode length   : {len(episode_regimes)} steps')
print(f'Final portfolio  : ${portfolio_values[-1]:,.2f}')
total_return = (portfolio_values[-1] - portfolio_values[0]) / portfolio_values[0]
print(f'Total return     : {total_return * 100:.2f}%')

## 5. Equity Curve

In [ ]:
from src.evaluation.visualizer import Visualizer

viz = Visualizer()
fig = viz.plot_equity_curve(portfolio_values, episode_regimes)
plt.show()

In [ ]:
fig = viz.plot_drawdown(portfolio_values)
plt.show()

In [ ]:
fig = viz.plot_regime_distribution(episode_regimes)
plt.show()

## 6. (Optional) Backtester with a trivial hold-all agent

In [ ]:
import numpy as np
from src.evaluation.backtester import Backtester

class HoldFirstStrategy:
    """Always allocates 100% to strategy 0 (momentum)."""
    def act(self, obs):
        w = np.zeros(4, dtype=np.float32)
        w[0] = 1.0
        return w

env2 = TradingEnv(data, lookback_window=20)
bt = Backtester(env2, HoldFirstStrategy())
results = bt.run(n_episodes=1)

print(f"Total Return     : {results['total_return'] * 100:.2f}%")
print(f"Sharpe Ratio     : {results['sharpe_ratio']:.3f}")
print(f"Max Drawdown     : {results['max_drawdown'] * 100:.2f}%")
print(f"Benchmark (B&H)  : {results['benchmark_return'] * 100:.2f}%")